# 19 · Advanced Window Functions

Module 10 introduced windows. Now the powerful parts engineers rely on:
- **frame clauses**: `ROWS`/`RANGE BETWEEN ... AND ...`
- **moving averages** and windowed aggregates
- `NTILE` (quantile buckets)
- `FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`
- distribution: `PERCENT_RANK`, `CUME_DIST`

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Frames: `ROWS BETWEEN ... AND ...`
A window can be narrowed to a **frame** relative to the current row. This is what
powers moving averages. Here: a 3-order moving average of order totals
(current row + the 2 before it).

In [ ]:
%%sql
WITH t AS (
    SELECT o.order_id, o.order_date, SUM(oi.quantity * oi.unit_price) AS total
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.order_date
)
SELECT order_date, ROUND(total, 2) AS total,
       ROUND(AVG(total) OVER (ORDER BY order_date, order_id
                              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM t
ORDER BY order_date, order_id;

### `ROWS` vs `RANGE`
`ROWS` counts a fixed number of physical rows; `RANGE` groups rows with the same
`ORDER BY` value into the frame together. For running totals with unique dates
they behave the same, but with ties they differ — reach for `ROWS` when you want
exactly *N* rows.

## `NTILE` — split into buckets
Divide products into 4 price quartiles (1 = cheapest quartile):

In [ ]:
%%sql
SELECT product_name, unit_price,
       NTILE(4) OVER (ORDER BY unit_price) AS price_quartile
FROM products
ORDER BY unit_price;

## `FIRST_VALUE` / `LAST_VALUE`
Show, next to each product, the most and least expensive product in its category.
Note the explicit full frame on `LAST_VALUE` — without it the frame ends at the
current row and you'd get the wrong answer (a very common bug).

In [ ]:
%%sql
SELECT category_id, product_name, unit_price,
       FIRST_VALUE(product_name) OVER w AS priciest_in_cat,
       LAST_VALUE(product_name)  OVER w AS cheapest_in_cat
FROM products
WINDOW w AS (PARTITION BY category_id ORDER BY unit_price DESC
             ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
ORDER BY category_id, unit_price DESC;

*(The `WINDOW w AS (...)` clause names a window so multiple functions can reuse it — handy when several columns share the same frame.)*

## Distribution: `PERCENT_RANK` & `CUME_DIST`
Where does each product's price sit in the overall distribution?
- `PERCENT_RANK` → relative rank from 0 to 1
- `CUME_DIST` → fraction of rows at or below this value

In [ ]:
%%sql
SELECT product_name, unit_price,
       ROUND(PERCENT_RANK() OVER (ORDER BY unit_price), 2) AS pct_rank,
       ROUND(CUME_DIST()   OVER (ORDER BY unit_price), 2) AS cume_dist
FROM products
ORDER BY unit_price DESC
LIMIT 10;

## Practice

**✏️ Exercise 1.** Compute a running (cumulative) count of customers over signup_date order.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, signup_date,
       COUNT(*) OVER (ORDER BY signup_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS customers_so_far
FROM customers
ORDER BY signup_date;

**✏️ Exercise 2.** Split employees into 3 salary bands using NTILE(3) (1 = lowest band).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, salary, NTILE(3) OVER (ORDER BY salary) AS salary_band
FROM employees
ORDER BY salary;

### ✅ Recap
Frames (`ROWS`/`RANGE BETWEEN`) turn windows into moving calculations; `NTILE`
buckets rows; `FIRST_VALUE`/`LAST_VALUE` grab boundary values (mind the frame!);
`PERCENT_RANK`/`CUME_DIST` describe distributions.

**Next:** `20_nulls_and_three_valued_logic.ipynb`.